# Feature engineering for propensity upsell scoring

This notebook loads the marketing campaign data generated by `generate_data.py`,
creates behavioral and interaction features, encodes categoricals, scales
numeric features, and examines correlations with the upsell response target.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.insert(0, '..')

from src.data_loader import load_data, engineer_features, get_feature_columns
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', 30)
plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')

## 1. Load customer data

In [ ]:
df = load_data('../data/marketing_campaign.csv')
print(f'Shape: {df.shape}')
print(f'Response rate: {df["responded"].mean():.3f}')
print(f'\nColumns: {list(df.columns)}')
df.head()

In [ ]:
# Basic statistics for numeric columns
df.describe().round(2)

## 2. Create behavioral features

The `engineer_features()` function from `data_loader.py` creates several
derived features that capture customer engagement and upsell potential.

In [ ]:
df = engineer_features(df)

new_cols = ['revenue_per_tenure', 'usage_intensity', 'service_count',
            'upsell_headroom', 'plan_encoded', 'channel_encoded',
            'income_x_tenure', 'data_x_streaming']

print('Engineered feature descriptions:')
print('  revenue_per_tenure:  monthly_spend / tenure_months (spending efficiency)')
print('  usage_intensity:     normalized composite of data, calls, sms usage')
print('  service_count:       sum of streaming + international + device insurance')
print('  upsell_headroom:     how many plan tiers can customer move up (0-2)')
print('  plan_encoded:        Basic=0, Standard=1, Premium=2')
print('  channel_encoded:     Email=0, SMS=1, App=2, Direct mail=3')
print('  income_x_tenure:     interaction: income * tenure_months / 1M')
print('  data_x_streaming:    interaction: data_usage * has_streaming')
print()
df[new_cols].describe().round(3)

In [ ]:
# Distribution of key behavioral features
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()

for ax, col in zip(axes, new_cols):
    if col in df.columns:
        df[col].hist(bins=30, ax=ax, alpha=0.7, color='steelblue', edgecolor='black')
        ax.set_title(col)
        ax.set_ylabel('Count')

plt.suptitle('Distribution of engineered features', fontsize=13)
plt.tight_layout()
plt.show()

## 3. Encode categorical variables

The `engineer_features()` function handles encoding:
- `current_plan` is mapped to ordinal values (Basic=0, Standard=1, Premium=2)
- `channel_preference` is mapped to numeric codes

Both preserve ordinal relationships that the models can exploit.

In [ ]:
# Verify encoding
print('Plan encoding:')
print(df.groupby('current_plan')['plan_encoded'].first())
print()
print('Channel encoding:')
print(df.groupby('channel_preference')['channel_encoded'].first())

# Response rate by plan
plan_resp = df.groupby('current_plan')['responded'].agg(['mean', 'count'])
plan_resp.columns = ['response_rate', 'count']
print('\nResponse rate by plan:')
print(plan_resp.round(3))

## 4. Feature scaling

In [ ]:
feature_cols = get_feature_columns()
print(f'Feature columns for modeling ({len(feature_cols)}):')
for i, col in enumerate(feature_cols, 1):
    print(f'  {i:2d}. {col}')

X = df[feature_cols].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f'\nScaled feature matrix shape: {X_scaled.shape}')
print(f'Column means after scaling: {np.round(X_scaled.mean(axis=0), 4)[:5]} ...')
print(f'Column stds after scaling:  {np.round(X_scaled.std(axis=0), 4)[:5]} ...')

## 5. Correlation analysis

In [ ]:
# Correlation with response target
corr_with_target = df[feature_cols + ['responded']].corr()['responded'].drop('responded')
corr_sorted = corr_with_target.abs().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 7))
colors = ['#FF5722' if corr_with_target[f] > 0 else '#2196F3'
          for f in corr_sorted.index]
ax.barh(range(len(corr_sorted)),
        [corr_with_target[f] for f in corr_sorted.index],
        color=colors)
ax.set_yticks(range(len(corr_sorted)))
ax.set_yticklabels(corr_sorted.index)
ax.set_xlabel('Correlation with responded')
ax.set_title('Feature correlations with upsell response')
ax.axvline(0, color='black', linewidth=0.5)
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

In [ ]:
# Full correlation heatmap
corr_matrix = df[feature_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, ax=ax, linewidths=0.3,
            annot_kws={'size': 7})
ax.set_title('Inter-feature correlation matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Identify highly correlated pairs
threshold = 0.7
high_corr = []
for i in range(len(feature_cols)):
    for j in range(i + 1, len(feature_cols)):
        c = abs(corr_matrix.iloc[i, j])
        if c > threshold:
            high_corr.append((
                feature_cols[i], feature_cols[j], round(corr_matrix.iloc[i, j], 3)
            ))

if high_corr:
    print(f'Feature pairs with |correlation| > {threshold}:')
    for f1, f2, c in high_corr:
        print(f'  {f1} <-> {f2}: {c:+.3f}')
else:
    print(f'No feature pairs exceed |correlation| > {threshold}')

## 6. Response rate by feature segments

Examining response rates across feature segments helps validate
that our features capture meaningful variation in upsell propensity.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# By plan
plan_rates = df.groupby('current_plan')['responded'].mean()
axes[0].bar(plan_rates.index, plan_rates.values, color='steelblue')
axes[0].set_ylabel('Response rate')
axes[0].set_title('Response rate by plan')

# By upsell headroom
headroom_rates = df.groupby('upsell_headroom')['responded'].mean()
axes[1].bar(headroom_rates.index.astype(str), headroom_rates.values, color='coral')
axes[1].set_ylabel('Response rate')
axes[1].set_title('Response rate by upsell headroom')

# By previous response
prev_rates = df.groupby('previous_upsell_response')['responded'].mean()
axes[2].bar(['No prior response', 'Prior responder'], prev_rates.values, color='#4CAF50')
axes[2].set_ylabel('Response rate')
axes[2].set_title('Response rate by prior behavior')

plt.tight_layout()
plt.show()

## Summary

The feature engineering pipeline produces 19 features from the raw campaign data:

- **Behavioral features** (revenue_per_tenure, usage_intensity, service_count)
  capture customer engagement patterns
- **Upsell headroom** directly measures upgrade potential
- **Interaction features** (income_x_tenure, data_x_streaming) capture
  combined effects of customer attributes
- **Previous upsell response** is the strongest single predictor,
  confirming that past behavior predicts future response

The feature set is ready for modeling in `03_modeling.ipynb`.